# PostgreSQL Stored Procedures

This notebook introduces stored procedures and functions in PostgreSQL using the `sequencing_qc`
database from the SQL Warmups exercise.

**Topics covered:**
1. Scalar functions
2. Table-valued functions
3. Functions with OUT parameters
4. Stored procedures with transaction control
5. Function overloading
6. Trigger functions
7. Exception handling
8. Calling routines from Python

## 1. Setup and Connection

In [ ]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text

DB_USER = 'shanebrubaker'  # Change to your username
DB_PASSWORD = ''
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'sequencing_qc'

engine = create_engine(f'postgresql://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

def get_conn():
    return psycopg2.connect(
        dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
        host=DB_HOST, port=DB_PORT
    )

def exec_ddl(sql):
    """Execute DDL (CREATE / DROP) statements that need autocommit."""
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql)
    cur.close()
    conn.close()

def query(sql, params=None):
    """Return a DataFrame for a SELECT query."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

def call(sql, params=None):
    """Execute a CALL statement. Uses autocommit so procedures can COMMIT/ROLLBACK internally."""
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql, params)
    cur.close()
    conn.close()

# Test connection
try:
    with engine.connect() as conn:
        print('Connected to', DB_NAME)
except Exception as e:
    print('Connection failed:', e)

In [ ]:
## Troubleshooting for Stuck connections - ONLY RUN IF NEEDED ##
# Terminate all other connections to this database
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("""
    SELECT pg_terminate_backend(pid)
        FROM pg_stat_activity
       WHERE datname = 'sequencing_qc'
         AND pid <> pg_backend_pid();
  """)
rows = cur.fetchall()
print(f'Terminated {len(rows)} stale connection(s)')
cur.close()
conn.close()

#  This kills any idle/stuck backends holding locks without touching the data or requiring a restart. Then re-run from
#  the top.

## Re-run the Cell Above after running this ##

---
## 2. Scalar Functions

A **scalar function** takes zero or more arguments and returns a single value.
They can be used anywhere an expression is valid in a query.

### Key syntax
```sql
CREATE OR REPLACE FUNCTION function_name(param type, ...)
RETURNS return_type
LANGUAGE plpgsql
AS $$
DECLARE
    -- local variable declarations
BEGIN
    -- function body
    RETURN value;
END;
$$;
```

In [ ]:
# Example 2a: Compute mapping rate (%) from a run_id
exec_ddl("""
CREATE OR REPLACE FUNCTION get_mapping_rate(p_run_id VARCHAR)
RETURNS NUMERIC
LANGUAGE plpgsql
AS $$
DECLARE
    v_total   BIGINT;
    v_mapped  BIGINT;
BEGIN
    SELECT total_reads, mapped_reads
      INTO v_total, v_mapped
      FROM qc_metrics
     WHERE run_id = p_run_id;

    IF v_total IS NULL OR v_total = 0 THEN
        RETURN NULL;
    END IF;

    RETURN ROUND((v_mapped::NUMERIC / v_total) * 100, 2);
END;
$$;
""")
print('Function get_mapping_rate created.')

In [ ]:
# Call the scalar function in a SELECT
df = query("""
    SELECT run_id,
           get_mapping_rate(run_id) AS mapping_rate_pct
      FROM sequencing_runs
  ORDER BY mapping_rate_pct DESC
  LIMIT 10;
""")
display(df)

In [ ]:
# Example 2b: Classify coverage depth into a category label
exec_ddl("""
CREATE OR REPLACE FUNCTION coverage_category(p_coverage NUMERIC)
RETURNS VARCHAR
LANGUAGE plpgsql
AS $$
BEGIN
    RETURN CASE
        WHEN p_coverage < 30  THEN 'Low (<30x)'
        WHEN p_coverage < 80  THEN 'Medium (30-80x)'
        WHEN p_coverage < 120 THEN 'High (80-120x)'
        ELSE                       'Very High (>120x)'
    END;
END;
$$;
""")

df = query("""
    SELECT run_id,
           mean_coverage,
           coverage_category(mean_coverage) AS category
      FROM qc_metrics
  ORDER BY mean_coverage DESC;
""")
display(df)

---
## 3. Table-Valued Functions

A **table-valued function** returns a set of rows instead of a single value.
Use `RETURNS TABLE(col type, ...)` or `RETURNS SETOF type`.
Call them with `SELECT * FROM function_name(args)`.

In [ ]:
# Example 3a: Return full QC summary for all runs belonging to one patient
exec_ddl("""
CREATE OR REPLACE FUNCTION get_patient_qc(p_patient_id VARCHAR)
RETURNS TABLE(
    run_id         VARCHAR,
    sequencer      VARCHAR,
    sample_type    VARCHAR,
    mean_coverage  NUMERIC,
    mean_quality   NUMERIC,
    pass_filter    BOOLEAN
)
LANGUAGE plpgsql
AS $$
BEGIN
    RETURN QUERY
    SELECT  sr.run_id,
            sr.sequencer,
            s.sample_type,
            qm.mean_coverage,
            qm.mean_quality_score,
            qm.pass_filter
      FROM  patients p
      JOIN  samples s          ON p.patient_id  = s.patient_id
      JOIN  sequencing_runs sr ON s.sample_id   = sr.sample_id
      JOIN  qc_metrics qm      ON sr.run_id     = qm.run_id
     WHERE  p.patient_id = p_patient_id
  ORDER BY  sr.run_id;
END;
$$;
""")
print('Function get_patient_qc created.')

In [ ]:
# Pick the first patient and call the table-valued function
first_patient = query("SELECT patient_id FROM patients LIMIT 1;").iloc[0, 0]
print('Querying QC for patient:', first_patient)

df = query(f"SELECT * FROM get_patient_qc('{first_patient}');")
display(df)

In [ ]:
# Example 3b: Return runs whose coverage falls outside a user-supplied range
exec_ddl("""
CREATE OR REPLACE FUNCTION runs_outside_coverage(p_min NUMERIC, p_max NUMERIC)
RETURNS TABLE(
    run_id        VARCHAR,
    mean_coverage NUMERIC,
    flag          VARCHAR
)
LANGUAGE plpgsql
AS $$
BEGIN
    RETURN QUERY
    SELECT  qc_metrics.run_id,
            qc_metrics.mean_coverage,
            CASE
                WHEN qc_metrics.mean_coverage < p_min THEN 'BELOW MIN'
                ELSE 'ABOVE MAX'
            END::VARCHAR
      FROM  qc_metrics
     WHERE  qc_metrics.mean_coverage < p_min
        OR  qc_metrics.mean_coverage > p_max
  ORDER BY  qc_metrics.mean_coverage;
END;
$$;
""")

df = query("SELECT * FROM runs_outside_coverage(40, 120);")
display(df)

---
## 4. Functions with OUT Parameters

`OUT` parameters let a function return multiple named values without declaring a composite type.
The function returns a single row whose columns match the `OUT` parameters.

In [ ]:
# Example 4a: Return min / avg / max coverage in a single call
exec_ddl("""
CREATE OR REPLACE FUNCTION coverage_stats(
    OUT min_cov  NUMERIC,
    OUT avg_cov  NUMERIC,
    OUT max_cov  NUMERIC
)
LANGUAGE plpgsql
AS $$
BEGIN
    SELECT MIN(mean_coverage),
           ROUND(AVG(mean_coverage), 2),
           MAX(mean_coverage)
      INTO min_cov, avg_cov, max_cov
      FROM qc_metrics;
END;
$$;
""")

df = query("SELECT * FROM coverage_stats();")
display(df)

In [ ]:
# Example 4b: Per-patient summary with OUT parameters
exec_ddl("""
CREATE OR REPLACE FUNCTION patient_summary(
    p_patient_id   IN  VARCHAR,
    OUT total_runs INT,
    OUT passed_runs INT,
    OUT avg_coverage NUMERIC
)
LANGUAGE plpgsql
AS $$
BEGIN
    SELECT  COUNT(*),
            SUM(CASE WHEN qm.pass_filter THEN 1 ELSE 0 END),
            ROUND(AVG(qm.mean_coverage), 2)
      INTO  total_runs, passed_runs, avg_coverage
      FROM  patients p
      JOIN  samples s          ON p.patient_id  = s.patient_id
      JOIN  sequencing_runs sr ON s.sample_id   = sr.sample_id
      JOIN  qc_metrics qm      ON sr.run_id     = qm.run_id
     WHERE  p.patient_id = p_patient_id;
END;
$$;
""")

# Call it for every patient
df = query("""
    SELECT p.patient_id,
           p.diagnosis,
           (patient_summary(p.patient_id)).*
      FROM patients p
  ORDER BY p.patient_id;
""")
display(df)

---
## 5. Stored Procedures (`CREATE PROCEDURE`)

Unlike functions, **procedures** are called with `CALL` and can contain explicit
`COMMIT` / `ROLLBACK` statements — making them suitable for multi-step data
modification workflows.

Procedures cannot be used inside a `SELECT`; they must be invoked with `CALL`.

In [ ]:
# First, create a QC audit log table to give the procedures something to write to
exec_ddl("""
DROP TABLE IF EXISTS qc_audit_log;

CREATE TABLE qc_audit_log (
    log_id      SERIAL PRIMARY KEY,
    run_id      VARCHAR(20),
    action      VARCHAR(50),
    old_status  BOOLEAN,
    new_status  BOOLEAN,
    changed_at  TIMESTAMP DEFAULT NOW(),
    changed_by  VARCHAR(50)
);
""")
print('Table qc_audit_log created.')

In [ ]:
# Example 5a: Procedure that overrides the pass_filter flag and logs the change
exec_ddl("""
CREATE OR REPLACE PROCEDURE update_qc_status(
    p_run_id    VARCHAR,
    p_new_flag  BOOLEAN,
    p_changed_by VARCHAR DEFAULT 'system'
)
LANGUAGE plpgsql
AS $$
DECLARE
    v_old_flag BOOLEAN;
BEGIN
    -- Read current status
    SELECT pass_filter INTO v_old_flag
      FROM qc_metrics
     WHERE run_id = p_run_id;

    IF NOT FOUND THEN
        RAISE EXCEPTION 'Run % not found in qc_metrics', p_run_id;
    END IF;

    -- Update the flag
    UPDATE qc_metrics
       SET pass_filter = p_new_flag
     WHERE run_id = p_run_id;

    -- Write audit record
    INSERT INTO qc_audit_log (run_id, action, old_status, new_status, changed_by)
    VALUES (p_run_id, 'MANUAL_OVERRIDE', v_old_flag, p_new_flag, p_changed_by);

    COMMIT;  -- Procedures can commit inside the body
END;
$$;
""")
print('Procedure update_qc_status created.')

In [ ]:
# Pick an arbitrary run and flip its QC flag
sample_run = query("SELECT run_id, pass_filter FROM qc_metrics LIMIT 1;").iloc[0]
run_id     = sample_run['run_id']
old_flag   = sample_run['pass_filter']
new_flag   = not old_flag

print(f'Changing run {run_id}: {old_flag} → {new_flag}')
call("CALL update_qc_status(%s, %s, %s);", (run_id, new_flag, 'notebook_demo'))

# Verify
df = query(f"SELECT * FROM qc_audit_log WHERE run_id = '{run_id}';")
display(df)

In [ ]:
# Example 5b: Procedure that batch-flags all runs below a coverage threshold as failed
exec_ddl("""
CREATE OR REPLACE PROCEDURE flag_low_coverage_runs(
    p_min_coverage  NUMERIC,
    p_changed_by    VARCHAR DEFAULT 'system'
)
LANGUAGE plpgsql
AS $$
DECLARE
    v_rec   RECORD;
    v_count INT := 0;
BEGIN
    FOR v_rec IN
        SELECT run_id, pass_filter
          FROM qc_metrics
         WHERE mean_coverage < p_min_coverage
           AND pass_filter = TRUE
    LOOP
        UPDATE qc_metrics
           SET pass_filter = FALSE
         WHERE run_id = v_rec.run_id;

        INSERT INTO qc_audit_log (run_id, action, old_status, new_status, changed_by)
        VALUES (v_rec.run_id, 'AUTO_FAIL_LOW_COV', TRUE, FALSE, p_changed_by);

        v_count := v_count + 1;
    END LOOP;

    RAISE NOTICE 'Flagged % run(s) with coverage below %x', v_count, p_min_coverage;
    COMMIT;
END;
$$;
""")

call("CALL flag_low_coverage_runs(50, 'notebook_demo');")

df = query("SELECT * FROM qc_audit_log ORDER BY changed_at DESC;")
display(df)

---
## 6. Function Overloading

PostgreSQL supports **function overloading**: you can define multiple functions
with the same name as long as they have different parameter signatures.
The correct version is chosen at call time based on the argument types.

In [ ]:
# Version 1: accepts a single run_id VARCHAR
exec_ddl("""
CREATE OR REPLACE FUNCTION qc_summary(p_run_id VARCHAR)
RETURNS TEXT
LANGUAGE plpgsql
AS $$
DECLARE
    v_cov  NUMERIC;
    v_qual NUMERIC;
    v_pass BOOLEAN;
BEGIN
    SELECT mean_coverage, mean_quality_score, pass_filter
      INTO v_cov, v_qual, v_pass
      FROM qc_metrics
     WHERE run_id = p_run_id;

    RETURN format('Run %s | Coverage: %sx | Quality: %s | Status: %s',
                  p_run_id, v_cov, v_qual,
                  CASE WHEN v_pass THEN 'PASS' ELSE 'FAIL' END);
END;
$$;
""")

# Version 2: accepts a run_id INTEGER (metric_id) — different type, same name
exec_ddl("""
CREATE OR REPLACE FUNCTION qc_summary(p_metric_id INTEGER)
RETURNS TEXT
LANGUAGE plpgsql
AS $$
DECLARE
    v_run_id VARCHAR;
    v_cov    NUMERIC;
    v_qual   NUMERIC;
    v_pass   BOOLEAN;
BEGIN
    SELECT run_id, mean_coverage, mean_quality_score, pass_filter
      INTO v_run_id, v_cov, v_qual, v_pass
      FROM qc_metrics
     WHERE metric_id = p_metric_id;

    RETURN format('Metric %s (Run %s) | Coverage: %sx | Quality: %s | Status: %s',
                  p_metric_id, v_run_id, v_cov, v_qual,
                  CASE WHEN v_pass THEN 'PASS' ELSE 'FAIL' END);
END;
$$;
""")
print('Two overloads of qc_summary created.')

In [ ]:
# PostgreSQL picks the right overload based on argument type
df_by_name = query("SELECT qc_summary('RUN0001') AS summary;")
df_by_id   = query("SELECT qc_summary(1) AS summary;")

print('Called with VARCHAR (run_id):')
display(df_by_name)

print('Called with INTEGER (metric_id):')
display(df_by_id)

---
## 7. Trigger Functions

A **trigger function** is a special function that returns `TRIGGER` and is
automatically invoked by PostgreSQL when rows are inserted, updated, or deleted.

Inside the trigger body, PostgreSQL provides special variables:
- `NEW` — the new row (available on INSERT / UPDATE)
- `OLD` — the old row (available on UPDATE / DELETE)
- `TG_OP` — the operation: `'INSERT'`, `'UPDATE'`, or `'DELETE'`

In [ ]:
# Create a dedicated audit table for sample changes
exec_ddl("""
DROP TABLE IF EXISTS samples_audit;

CREATE TABLE samples_audit (
    audit_id        SERIAL PRIMARY KEY,
    operation       VARCHAR(10),
    sample_id       VARCHAR(20),
    old_concentration NUMERIC,
    new_concentration NUMERIC,
    changed_at      TIMESTAMP DEFAULT NOW()
);
""")
print('Table samples_audit created.')

In [ ]:
# Step 1: Create the trigger function
exec_ddl("""
CREATE OR REPLACE FUNCTION audit_sample_changes()
RETURNS TRIGGER
LANGUAGE plpgsql
AS $$
BEGIN
    IF TG_OP = 'INSERT' THEN
        INSERT INTO samples_audit (operation, sample_id, old_concentration, new_concentration)
        VALUES ('INSERT', NEW.sample_id, NULL, NEW.dna_concentration);

    ELSIF TG_OP = 'UPDATE' THEN
        -- Only log if concentration actually changed
        IF OLD.dna_concentration IS DISTINCT FROM NEW.dna_concentration THEN
            INSERT INTO samples_audit (operation, sample_id, old_concentration, new_concentration)
            VALUES ('UPDATE', NEW.sample_id, OLD.dna_concentration, NEW.dna_concentration);
        END IF;

    ELSIF TG_OP = 'DELETE' THEN
        INSERT INTO samples_audit (operation, sample_id, old_concentration, new_concentration)
        VALUES ('DELETE', OLD.sample_id, OLD.dna_concentration, NULL);
    END IF;

    RETURN NEW;  -- Required for row-level triggers; ignored for AFTER triggers
END;
$$;
""")
print('Trigger function audit_sample_changes created.')

In [ ]:
# Step 2: Attach the trigger to the samples table
exec_ddl("""
DROP TRIGGER IF EXISTS trg_audit_samples ON samples;

CREATE TRIGGER trg_audit_samples
AFTER INSERT OR UPDATE OR DELETE ON samples
FOR EACH ROW
EXECUTE FUNCTION audit_sample_changes();
""")
print('Trigger trg_audit_samples attached to samples.')

In [ ]:
# Test the trigger with an UPDATE
conn = get_conn()
cur  = conn.cursor()

cur.execute("SELECT sample_id, dna_concentration FROM samples LIMIT 1;")
sid, old_conc = cur.fetchone()
new_conc = round(float(old_conc) + 10.0, 2)

print(f'Updating sample {sid}: {old_conc} → {new_conc} ng/µL')

cur.execute(
    "UPDATE samples SET dna_concentration = %s WHERE sample_id = %s;",
    (new_conc, sid)
)
conn.commit()
cur.close()
conn.close()

# The trigger should have written a row automatically
df = query("SELECT * FROM samples_audit ORDER BY changed_at DESC LIMIT 5;")
display(df)

In [ ]:
# Test the trigger with an INSERT of a temporary sample
conn = get_conn()
cur  = conn.cursor()
cur.execute("""
    INSERT INTO samples (sample_id, patient_id, sample_type, collection_date,
                         tissue_source, dna_concentration)
    VALUES ('S9999', 'PT0001', 'Tumor', CURRENT_DATE, 'Blood', 75.50);
""")
conn.commit()

# Clean up: delete the temporary row (trigger fires on DELETE too)
cur.execute("DELETE FROM samples WHERE sample_id = 'S9999';")
conn.commit()
cur.close()
conn.close()

df = query("SELECT * FROM samples_audit ORDER BY changed_at DESC LIMIT 5;")
display(df)

---
## 8. Exception Handling

PL/pgSQL supports structured exception handling with `BEGIN … EXCEPTION … END`.
You can catch specific error codes or the generic `OTHERS` catch-all.
Use `RAISE` to emit `NOTICE`, `WARNING`, or `EXCEPTION` messages.

In [ ]:
# Example 8a: Safe wrapper that returns NULL instead of raising on missing run
exec_ddl("""
CREATE OR REPLACE FUNCTION safe_get_coverage(p_run_id VARCHAR)
RETURNS NUMERIC
LANGUAGE plpgsql
AS $$
DECLARE
    v_coverage NUMERIC;
BEGIN
    SELECT mean_coverage INTO STRICT v_coverage
      FROM qc_metrics
     WHERE run_id = p_run_id;

    RETURN v_coverage;

EXCEPTION
    WHEN NO_DATA_FOUND THEN
        RAISE WARNING 'Run % not found — returning NULL', p_run_id;
        RETURN NULL;

    WHEN TOO_MANY_ROWS THEN
        RAISE WARNING 'Multiple rows for run % — returning NULL', p_run_id;
        RETURN NULL;
END;
$$;
""")

# Valid run returns a value; made-up ID returns NULL
df = query("""
    SELECT safe_get_coverage('RUN0001')  AS existing_run,
           safe_get_coverage('RUN_FAKE') AS missing_run;
""")
display(df)

In [ ]:
# Example 8b: Procedure with a rollback on error
exec_ddl("""
CREATE OR REPLACE PROCEDURE safe_insert_sample(
    p_sample_id    VARCHAR,
    p_patient_id   VARCHAR,
    p_sample_type  VARCHAR,
    p_dna_conc     NUMERIC
)
LANGUAGE plpgsql
AS $$
BEGIN
    -- Validate concentration is positive
    IF p_dna_conc <= 0 THEN
        RAISE EXCEPTION 'dna_concentration must be positive, got: %', p_dna_conc
            USING ERRCODE = 'check_violation';
    END IF;

    INSERT INTO samples (sample_id, patient_id, sample_type,
                         collection_date, tissue_source, dna_concentration)
    VALUES (p_sample_id, p_patient_id, p_sample_type,
            CURRENT_DATE, 'Unknown', p_dna_conc);

    RAISE NOTICE 'Sample % inserted successfully.', p_sample_id;
    COMMIT;

EXCEPTION
    WHEN check_violation OR unique_violation THEN
        RAISE WARNING 'Insert of sample % failed: %', p_sample_id, SQLERRM;
        ROLLBACK;

    WHEN OTHERS THEN
        RAISE WARNING 'Unexpected error for sample %: %', p_sample_id, SQLERRM;
        ROLLBACK;
END;
$$;
""")
print('Procedure safe_insert_sample created.')

In [ ]:
count_before = query("SELECT COUNT(*) AS n FROM samples;").iloc[0, 0]

# This call should fail validation (negative concentration)
try:
    call("CALL safe_insert_sample('S8888', 'PT0001', 'Tumor', -5.0);")
except Exception as e:
    print('Expected error caught in Python:', e)

count_after = query("SELECT COUNT(*) AS n FROM samples;").iloc[0, 0]
print(f'Row count before: {count_before}, after: {count_after}  (should be equal)')

---
## 9. Calling Stored Routines from Python

This section shows practical patterns for invoking PostgreSQL functions and
procedures from Python code using both **psycopg2** and **SQLAlchemy**.

In [ ]:
# Pattern A: Call a scalar function and fetch its return value with psycopg2
conn = get_conn()
cur  = conn.cursor()
cur.execute("SELECT get_mapping_rate(%s);", ('RUN0003',))
rate = cur.fetchone()[0]
print(f'Mapping rate for RUN0003: {rate}%')
cur.close()
conn.close()

In [ ]:
# Pattern B: Call a table-valued function and load results into a DataFrame
patient_ids = query("SELECT patient_id FROM patients;")['patient_id'].tolist()

frames = []
for pid in patient_ids:
    df = query("SELECT * FROM get_patient_qc(:pid);", params={'pid': pid})
    df.insert(0, 'patient_id', pid)
    frames.append(df)

all_qc = pd.concat(frames, ignore_index=True)
print(f'Total rows across all patients: {len(all_qc)}')
display(all_qc.head(10))

In [ ]:
# Pattern C: Call a PROCEDURE with CALL via psycopg2
runs_to_flip = query("""
    SELECT run_id FROM qc_metrics WHERE pass_filter = FALSE LIMIT 3;
""")['run_id'].tolist()

conn = get_conn()
conn.autocommit = True  # Required: procedures with COMMIT can't run inside a transaction block
cur  = conn.cursor()
for run_id in runs_to_flip:
    cur.execute("CALL update_qc_status(%s, %s, %s);",
                (run_id, True, 'python_batch_repass'))
cur.close()
conn.close()

df = query("""
    SELECT * FROM qc_audit_log
     WHERE changed_by = 'python_batch_repass'
  ORDER BY changed_at DESC;
""")
display(df)

In [ ]:
# Pattern D: Inspect installed functions using the PostgreSQL catalog
df = query("""
    SELECT  p.proname                         AS function_name,
            pg_get_function_arguments(p.oid)  AS arguments,
            t.typname                         AS return_type,
            l.lanname                         AS language
      FROM  pg_proc p
      JOIN  pg_type t ON p.prorettype = t.oid
      JOIN  pg_language l ON p.prolang = l.oid
     WHERE  p.pronamespace = 'public'::regnamespace
       AND  l.lanname = 'plpgsql'
  ORDER BY  p.proname;
""")
display(df)

---
## 10. Cleanup (Optional)

Run the cell below to drop all objects created by this notebook.
The original `sequencing_qc` tables from the warmups exercise are **not** affected.

In [ ]:
cleanup_sql = """
-- Drop triggers first
DROP TRIGGER IF EXISTS trg_audit_samples ON samples;

-- Drop functions (all overloads)
DROP FUNCTION IF EXISTS get_mapping_rate(VARCHAR);
DROP FUNCTION IF EXISTS coverage_category(NUMERIC);
DROP FUNCTION IF EXISTS get_patient_qc(VARCHAR);
DROP FUNCTION IF EXISTS runs_outside_coverage(NUMERIC, NUMERIC);
DROP FUNCTION IF EXISTS coverage_stats();
DROP FUNCTION IF EXISTS patient_summary(VARCHAR);
DROP FUNCTION IF EXISTS qc_summary(VARCHAR);
DROP FUNCTION IF EXISTS qc_summary(INTEGER);
DROP FUNCTION IF EXISTS safe_get_coverage(VARCHAR);
DROP FUNCTION IF EXISTS audit_sample_changes();

-- Drop procedures
DROP PROCEDURE IF EXISTS update_qc_status(VARCHAR, BOOLEAN, VARCHAR);
DROP PROCEDURE IF EXISTS flag_low_coverage_runs(NUMERIC, VARCHAR);
DROP PROCEDURE IF EXISTS safe_insert_sample(VARCHAR, VARCHAR, VARCHAR, NUMERIC);

-- Drop extra tables
DROP TABLE IF EXISTS qc_audit_log;
DROP TABLE IF EXISTS samples_audit;
"""

exec_ddl(cleanup_sql)
print('All stored procedure objects removed.')